# Evaluating Tool-Calling Agents Without Real Tool Execution

## Overview

This notebook addresses one of the hardest problems in agentic AI evaluation: **how to rigorously evaluate an agent's tool-calling behavior without executing real tools.**

Real tool calls are expensive, slow, non-deterministic, and can have side effects (sending emails, modifying databases, charging credit cards). Yet tool calling is the core capability that separates an agent from a chatbot. If you can't evaluate it safely and cheaply, you can't iterate on it.

### The Three Decisions You're Evaluating

Every tool call involves three model decisions:

1. **Whether** to call a tool (vs. responding conversationally)
2. **Which** tool to call (tool selection)
3. **What** parameters to pass (parameter generation)

Each can be evaluated independently, at different cost points, with different techniques.

### The Evaluation Pyramid

We demonstrate **six approaches**, ordered from cheapest to most expensive:

| Layer | Approach | Cost | Speed | Best For |
|-------|----------|------|-------|----------|
| 1 | Static Trajectory Analysis | $0 | Instant | Analyzing production logs, regression testing |
| 2 | Schema Validation | $0 | Instant | Catching parameter hallucination, type errors |
| 3 | Mock Tool Evaluation | $ | Seconds | Testing live model decisions safely |
| 4 | LLM-as-Judge | $$ | Seconds | Subjective quality, edge cases |
| 5 | Multi-Turn Simulation | $$$ | Minutes | End-to-end goal achievement |
| 6 | Strands ToolSimulator + ActorSimulator | $$$$ | Minutes | Fully synthetic, stateful environment simulation |

Approaches 1-5 use the **Bedrock Converse API directly** (no agent framework required). Approach 6 demonstrates the **Strands Agents Evals SDK**, which provides LLM-powered tool and user simulation with stateful consistency — the most sophisticated approach available.

## 1. Setup

In [ ]:
!pip install boto3 pandas jsonschema --quiet
!pip install strands-agents strands-agents-evals --quiet

### Imports

In [ ]:
import boto3
import json
import re
import pandas as pd
from botocore.config import Config
from collections import defaultdict
from copy import deepcopy
import jsonschema

### Bedrock Client & Model Configuration

In [ ]:
# Bedrock client with reasonable timeouts for evaluation
bedrock = boto3.client(
    'bedrock-runtime',
    config=Config(
        connect_timeout=5,
        read_timeout=60,
        retries={"max_attempts": 2}
    )
)

# Models - both configurable
# Agent under test: the model whose tool-calling we're evaluating
AGENT_MODEL = "us.anthropic.claude-sonnet-4-20250514-v1:0"

# Judge model: evaluates the quality of tool-calling decisions
JUDGE_MODEL = "us.anthropic.claude-sonnet-4-20250514-v1:0"

print(f"Agent model: {AGENT_MODEL}")
print(f"Judge model: {JUDGE_MODEL}")

### Load Evaluation Data

In [ ]:
# Load tool definitions, test cases, and pre-recorded traces
with open('data/tool_definitions.json') as f:
    tool_defs = json.load(f)

with open('data/test_cases.json') as f:
    test_cases = json.load(f)

with open('data/recorded_traces.json') as f:
    recorded_traces = json.load(f)

print(f"Scenario: {tool_defs['scenario']}")
print(f"Tools available: {len(tool_defs['tools'])}")
print(f"Test cases: {len(test_cases)}")
print(f"Pre-recorded traces: {len(recorded_traces)}")

## 2. Understanding the Evaluation Scenario

We're evaluating a **customer support agent** for an e-commerce platform. The agent has access to 6 tools that interact with backend systems. Some test cases require tool calls, some don't — and the agent needs to know the difference.

The pre-recorded traces include both correct and incorrect behaviors, so we can test our evaluation methods on known-good and known-bad examples.

In [ ]:
# Display available tools
print("Available Tools")
print("=" * 70)
for tool in tool_defs['tools']:
    required = tool['parameters'].get('required', [])
    print(f"\n  {tool['name']}")
    print(f"    {tool['description'][:80]}")
    print(f"    Required params: {required}")

# Test case distribution
print(f"\n\nTest Case Distribution")
print("=" * 70)
categories = defaultdict(list)
for tc in test_cases:
    categories[tc['category']].append(tc['id'])
for cat, ids in categories.items():
    print(f"  {cat}: {len(ids)} cases ({', '.join(ids)})")

In [ ]:
# Examine the pre-recorded traces - note the intentional errors
print("Pre-Recorded Traces (with labels)")
print("=" * 70)
for trace in recorded_traces:
    tools_str = " -> ".join(trace['tools_called']) if trace['tools_called'] else "(no tools)"
    print(f"\n  {trace['trace_id']} [{trace['label']}]")
    print(f"    Input: {trace['input'][:60]}...")
    print(f"    Tools: {tools_str}")

## 3. Approach 1: Static Trajectory Analysis (Cost: $0)

The cheapest and fastest evaluation approach. Given pre-recorded traces (from production logs, staging tests, or synthetic runs), compute metrics **without making any API calls**.

This is your first line of defense — run it on every trace, every time. It catches:
- Wrong tool selection
- Missing tool calls
- Unnecessary tool calls
- Incorrect parameters
- Wrong tool sequence

### Metrics We'll Compute

- **Tool Selection Precision**: Of tools called, what fraction were correct?
- **Tool Selection Recall**: Of expected tools, what fraction were called?
- **F1 Score**: Harmonic mean of precision and recall
- **Call/No-Call Accuracy**: Did the agent correctly decide whether to use tools at all?
- **Parameter Accuracy**: Did parameters match expected values?
- **Sequence Match**: Were tools called in the correct order?

In [ ]:
def analyze_trace(trace, test_case):
    """Analyze a pre-recorded trace against its test case ground truth.

    Returns a dict of metrics computed purely from the trace data — no API calls needed.
    """
    results = {
        'trace_id': trace['trace_id'],
        'test_case_id': trace['test_case_id'],
        'label': trace['label'],
    }

    # --- Tool Selection Metrics ---
    expected_tools = set(test_case['expected_tools'])
    actual_tools = set(trace['tools_called'])

    tp = len(expected_tools & actual_tools)      # correctly called
    fp = len(actual_tools - expected_tools)       # shouldn't have called
    fn = len(expected_tools - actual_tools)       # should have called but didn't

    precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0  # 1.0 if nothing was called and nothing expected
    recall = tp / (tp + fn) if (tp + fn) > 0 else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    results['tool_precision'] = round(precision, 3)
    results['tool_recall'] = round(recall, 3)
    results['tool_f1'] = round(f1, 3)

    # --- Call/No-Call Decision ---
    should_call = test_case['should_call_tool']
    did_call = len(trace['tools_called']) > 0
    results['call_decision_correct'] = should_call == did_call

    # --- Sequence Match (order matters) ---
    results['sequence_match'] = trace['tools_called'] == test_case['expected_tools']

    # --- Parameter Accuracy ---
    param_scores = []
    for tool_name, expected_params in test_case['expected_params'].items():
        if tool_name in trace.get('params_used', {}):
            actual_params = trace['params_used'][tool_name]
            if expected_params:
                matching = sum(
                    1 for k, v in expected_params.items()
                    if k in actual_params and actual_params[k] == v
                )
                param_scores.append(matching / len(expected_params))
        elif expected_params:
            param_scores.append(0.0)  # tool wasn't called but should have been

    results['param_accuracy'] = round(
        sum(param_scores) / len(param_scores), 3
    ) if param_scores else None

    return results

In [ ]:
# Run static analysis on all pre-recorded traces
# First, build a lookup from test_case_id to test_case
tc_lookup = {tc['id']: tc for tc in test_cases}

trace_results = []
for trace in recorded_traces:
    tc = tc_lookup.get(trace['test_case_id'])
    if tc:
        result = analyze_trace(trace, tc)
        trace_results.append(result)

# Display results as a DataFrame
df_traces = pd.DataFrame(trace_results)
print("Static Trajectory Analysis Results")
print("=" * 90)
print(df_traces.to_string(index=False))

# Summary statistics
print(f"\n\nSummary")
print(f"  Call/No-Call Accuracy:  {df_traces['call_decision_correct'].mean():.1%}")
print(f"  Average Tool F1:       {df_traces['tool_f1'].mean():.3f}")
print(f"  Sequence Match Rate:   {df_traces['sequence_match'].mean():.1%}")
param_scores = df_traces['param_accuracy'].dropna()
print(f"  Average Param Accuracy:{param_scores.mean():.1%}" if len(param_scores) > 0 else "  Param Accuracy: N/A")

### Key Observations from Static Analysis

Look at the results above. The traces labeled `incorrect_*` should show clear metric failures:

- **trace-002** (`incorrect_unnecessary_tool_call`): Called `get_customer_profile` on a casual greeting. The call/no-call decision is wrong, and precision drops because of the false positive.
- **trace-006** (`incorrect_missing_parameter`): Called the right tool but missed the `warehouse_region` parameter. Tool F1 is fine, but parameter accuracy reveals the problem.
- **trace-008** (`incorrect_unnecessary_tool_call_with_hallucinated_params`): Called `lookup_order` with fabricated parameter `"POLICY"` on a general question. Both call decision and parameter accuracy fail.

**This is the power of static analysis**: three different failure modes, all caught for free.

## 4. Approach 2: Schema-Based Parameter Validation (Cost: $0)

JSON Schema validation catches an entire class of tool-calling errors programmatically:

- **Type errors**: passing a string where a number is expected
- **Missing required fields**: omitting mandatory parameters
- **Invalid enum values**: using values outside the allowed set
- **Pattern violations**: IDs that don't match the expected format (e.g., `ORD-XXXXX`)
- **Extra properties**: passing parameters the tool doesn't accept

This is especially important for **parameter hallucination** — when the model generates plausible-looking but invalid parameter values.

In [ ]:
def validate_tool_call(tool_name, params, tool_defs):
    """Validate tool parameters against the JSON Schema defined in tool_definitions.

    Returns a dict with validation results and specific errors.
    """
    # Find the tool definition
    tool_schema = next(
        (t for t in tool_defs['tools'] if t['name'] == tool_name), None
    )
    if not tool_schema:
        return {
            'tool': tool_name,
            'valid': False,
            'errors': [f"Unknown tool: {tool_name}"],
            'error_types': ['unknown_tool']
        }

    schema = tool_schema['parameters']
    errors = []
    error_types = []

    # Validate against JSON Schema
    validator = jsonschema.Draft7Validator(schema)
    for error in validator.iter_errors(params):
        errors.append(error.message)

        # Classify the error type
        if error.validator == 'required':
            error_types.append('missing_required')
        elif error.validator == 'type':
            error_types.append('wrong_type')
        elif error.validator == 'enum':
            error_types.append('invalid_enum')
        elif error.validator == 'pattern':
            error_types.append('pattern_violation')
        elif error.validator == 'additionalProperties':
            error_types.append('extra_property')
        else:
            error_types.append(error.validator)

    # Additional check: validate pattern for order_id fields manually
    # (jsonschema pattern validation may not be in all tool schemas)
    if 'order_id' in params:
        order_id = params['order_id']
        if not re.match(r'^ORD-\d{5}$', str(order_id)):
            errors.append(f"order_id '{order_id}' doesn't match pattern ORD-XXXXX")
            error_types.append('pattern_violation')

    return {
        'tool': tool_name,
        'valid': len(errors) == 0,
        'errors': errors,
        'error_types': error_types
    }

In [ ]:
# Validate all tool calls in the pre-recorded traces
print("Schema Validation Results")
print("=" * 90)

validation_results = []
for trace in recorded_traces:
    for tool_name in trace['tools_called']:
        params = trace['params_used'].get(tool_name, {})
        result = validate_tool_call(tool_name, params, tool_defs)
        result['trace_id'] = trace['trace_id']
        result['label'] = trace['label']
        validation_results.append(result)

        status = "PASS" if result['valid'] else "FAIL"
        print(f"  {trace['trace_id']} | {tool_name:25s} | {status}")
        if not result['valid']:
            for err in result['errors']:
                print(f"    -> {err}")

# Summary
total = len(validation_results)
valid = sum(1 for r in validation_results if r['valid'])
print(f"\nSchema Compliance: {valid}/{total} ({valid/total:.0%})")

# Error type breakdown
error_counts = defaultdict(int)
for r in validation_results:
    for et in r['error_types']:
        error_counts[et] += 1
if error_counts:
    print("\nError Type Breakdown:")
    for etype, count in sorted(error_counts.items(), key=lambda x: -x[1]):
        print(f"  {etype}: {count}")

### What Schema Validation Catches

Notice that **trace-008** fails schema validation — the agent called `lookup_order` with `order_id: "POLICY"`, which doesn't match the `ORD-XXXXX` pattern. This is a classic **parameter hallucination**: the model invented a plausible-but-invalid parameter value.

Schema validation is your cheapest defense against parameter hallucination. Run it on every trace, every time.

## 5. Approach 3: Mock Tool Evaluation (Cost: $ per API call)

This is where evaluation gets powerful. We run the **actual model live** using the Bedrock Converse API with real tool definitions, but instead of executing real tools, we return **mock responses**.

The model makes real decisions about:
- Whether to call a tool
- Which tool to call
- What parameters to pass

But no real side effects occur. The mock responses are deterministic and realistic enough for the model to reason about.

### How It Works

1. Send user input to the model via Bedrock Converse API with `toolConfig`
2. When the model returns `toolUse` blocks, intercept them
3. Return pre-defined mock responses as `toolResult`
4. Continue the conversation until the model gives a final text response
5. Compare the full trajectory against ground truth

In [ ]:
# Mock tool responses - realistic enough for the model to reason about
MOCK_RESPONSES = {
    "lookup_order": lambda params: {
        "order_id": params.get("order_id", "ORD-00000"),
        "status": "delivered",
        "items": ["Wireless Headphones", "Phone Case"],
        "delivered_date": "2026-04-01",
        "shipping_address": {"street": "123 Main St", "city": "Seattle", "state": "WA", "zip": "98101"},
        "total": "$89.99",
        "payment_status": "charged",
        "charges": [{"amount": "$89.99", "date": "2026-03-28"}]
    },
    "initiate_return": lambda params: {
        "return_auth": "RET-77201",
        "instructions": "Print the prepaid label and drop off at any UPS location within 14 days.",
        "refund_estimate": "5-7 business days after receipt"
    },
    "check_inventory": lambda params: {
        "product_id": params.get("product_id", "UNKNOWN"),
        "available": True,
        "quantity": 156,
        "warehouse": params.get("warehouse_region", "us-east"),
        "estimated_delivery": "2-3 business days"
    },
    "update_shipping_address": lambda params: {
        "status": "updated",
        "order_id": params.get("order_id", "ORD-00000"),
        "new_address": params.get("new_address", {}),
        "note": "Address updated successfully. Order has not yet shipped."
    },
    "get_customer_profile": lambda params: {
        "customer_id": params.get("customer_id", "UNKNOWN"),
        "name": "Jane Smith",
        "tier": "Gold",
        "member_since": "2023-01-15",
        "total_orders": 47,
        "preferences": {"notifications": "email", "language": "en"}
    },
    "escalate_to_human": lambda params: {
        "ticket": "ESC-3301",
        "estimated_wait": "5 minutes",
        "priority": params.get("priority", "medium"),
        "status": "queued"
    }
}

print(f"Mock responses defined for {len(MOCK_RESPONSES)} tools")

### Convert Tool Definitions to Bedrock Converse Format

In [ ]:
def to_converse_tool_config(tool_defs):
    """Convert our tool definitions to Bedrock Converse API toolConfig format."""
    tools = []
    for tool in tool_defs['tools']:
        tools.append({
            "toolSpec": {
                "name": tool['name'],
                "description": tool['description'],
                "inputSchema": {
                    "json": tool['parameters']
                }
            }
        })
    return {"tools": tools}

tool_config = to_converse_tool_config(tool_defs)
print(f"Converted {len(tool_config['tools'])} tools to Converse API format")
print("Tool names:", [t['toolSpec']['name'] for t in tool_config['tools']])

### The Mock Tool Evaluation Loop

This is the core function. It runs the Converse API in a loop: send message -> check for tool calls -> return mock response -> repeat until the model gives a final answer.

In [ ]:
SYSTEM_PROMPT = """You are a helpful customer support agent for an e-commerce platform.
Help customers with order inquiries, returns, account management, and product questions.
Use the available tools to look up information and take actions.
Only call tools when the customer's request requires accessing backend systems.
For general questions, greetings, or policy inquiries, respond directly without using tools."""

def run_with_mock_tools(user_input, max_turns=10):
    """Run the Bedrock Converse API with mock tool responses.

    Returns the full trajectory including all tool calls, parameters, and the final response.
    """
    messages = [{"role": "user", "content": [{"text": user_input}]}]
    trajectory = []

    for turn in range(max_turns):
        response = bedrock.converse(
            modelId=AGENT_MODEL,
            messages=messages,
            system=[{"text": SYSTEM_PROMPT}],
            toolConfig=tool_config
        )

        stop_reason = response['stopReason']
        assistant_content = response['output']['message']['content']
        messages.append({"role": "assistant", "content": assistant_content})

        # If the model is done talking, extract final text and return
        if stop_reason == 'end_turn':
            final_text = ""
            for block in assistant_content:
                if 'text' in block:
                    final_text += block['text']
            break

        # If the model wants to use tools, intercept and return mocks
        if stop_reason == 'tool_use':
            tool_results = []
            for block in assistant_content:
                if 'toolUse' in block:
                    tool_call = block['toolUse']
                    tool_name = tool_call['name']
                    tool_params = tool_call['input']

                    # Record the trajectory
                    trajectory.append({
                        'tool': tool_name,
                        'params': tool_params,
                        'turn': turn
                    })

                    # Return mock response instead of executing real tool
                    mock_fn = MOCK_RESPONSES.get(tool_name)
                    if mock_fn:
                        mock_result = mock_fn(tool_params)
                    else:
                        mock_result = {"error": f"Unknown tool: {tool_name}"}

                    tool_results.append({
                        "toolResult": {
                            "toolUseId": tool_call['toolUseId'],
                            "content": [{"json": mock_result}],
                            "status": "success"
                        }
                    })

            messages.append({"role": "user", "content": tool_results})
    else:
        final_text = "[Max turns reached]"

    return {
        'trajectory': trajectory,
        'tools_called': [t['tool'] for t in trajectory],
        'params_used': {t['tool']: t['params'] for t in trajectory},
        'final_output': final_text,
        'num_turns': turn + 1
    }

### Run a Single Test Case

Let's test with one example first to see the mock tool evaluation in action.

In [ ]:
# Run a single test case
test = test_cases[3]  # tc-004: multi-step lookup then return
print(f"Test Case: {test['name']}")
print(f"Input: {test['input']}")
print(f"Expected Tools: {test['expected_tools']}")
print()

result = run_with_mock_tools(test['input'])

print(f"Tools Called: {result['tools_called']}")
print(f"Turns: {result['num_turns']}")
print(f"\nTrajectory:")
for step in result['trajectory']:
    print(f"  Turn {step['turn']}: {step['tool']}({json.dumps(step['params'], indent=4)})")
print(f"\nFinal Response: {result['final_output'][:200]}...")

### Run All Test Cases with Mock Tools

In [ ]:
# Run all test cases and compare against ground truth
print("Mock Tool Evaluation: Running all test cases...")
print("=" * 90)

mock_results = []
for tc in test_cases:
    try:
        result = run_with_mock_tools(tc['input'])

        # Compare against ground truth
        expected_tools = set(tc['expected_tools'])
        actual_tools = set(result['tools_called'])

        tp = len(expected_tools & actual_tools)
        fp = len(actual_tools - expected_tools)
        fn = len(expected_tools - actual_tools)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 1.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        # Call/no-call decision
        should_call = tc['should_call_tool']
        did_call = len(result['tools_called']) > 0

        # Parameter accuracy
        param_scores = []
        for tool_name, expected_params in tc['expected_params'].items():
            if tool_name in result['params_used']:
                actual_params = result['params_used'][tool_name]
                if expected_params:
                    matching = sum(
                        1 for k, v in expected_params.items()
                        if k in actual_params and actual_params[k] == v
                    )
                    param_scores.append(matching / len(expected_params))
            elif expected_params:
                param_scores.append(0.0)

        # Schema validation on actual calls
        schema_valid = all(
            validate_tool_call(t['tool'], t['params'], tool_defs)['valid']
            for t in result['trajectory']
        ) if result['trajectory'] else True

        mock_results.append({
            'test_id': tc['id'],
            'name': tc['name'][:30],
            'category': tc['category'],
            'expected': tc['expected_tools'],
            'actual': result['tools_called'],
            'f1': round(f1, 3),
            'call_correct': should_call == did_call,
            'param_acc': round(sum(param_scores)/len(param_scores), 3) if param_scores else None,
            'schema_valid': schema_valid,
            'seq_match': result['tools_called'] == tc['expected_tools']
        })

        status = "PASS" if f1 == 1.0 and (should_call == did_call) else "FAIL" if f1 < 0.5 or (should_call != did_call) else "PARTIAL"
        print(f"  {tc['id']} [{status:7s}] {tc['name'][:35]:35s} | F1={f1:.2f} | Call={'OK' if should_call == did_call else 'WRONG'}")

    except Exception as e:
        print(f"  {tc['id']} [ERROR  ] {tc['name'][:35]:35s} | {str(e)[:50]}")
        mock_results.append({
            'test_id': tc['id'], 'name': tc['name'][:30], 'category': tc['category'],
            'expected': tc['expected_tools'], 'actual': ['ERROR'],
            'f1': 0, 'call_correct': False, 'param_acc': None,
            'schema_valid': False, 'seq_match': False
        })

print()

In [ ]:
# Summary of mock tool evaluation results
df_mock = pd.DataFrame(mock_results)

print("Mock Tool Evaluation Summary")
print("=" * 70)
print(f"  Overall Tool F1:          {df_mock['f1'].mean():.3f}")
print(f"  Call/No-Call Accuracy:     {df_mock['call_correct'].mean():.1%}")
print(f"  Sequence Match Rate:       {df_mock['seq_match'].mean():.1%}")
print(f"  Schema Compliance:         {df_mock['schema_valid'].mean():.1%}")
param_scores = df_mock['param_acc'].dropna()
if len(param_scores) > 0:
    print(f"  Parameter Accuracy:        {param_scores.mean():.1%}")

print(f"\nResults by Category:")
for cat, group in df_mock.groupby('category'):
    print(f"  {cat:25s} | F1={group['f1'].mean():.2f} | Call Acc={group['call_correct'].mean():.0%}")

## 6. Approach 4: LLM-as-Judge Trajectory Evaluation (Cost: $$)

Programmatic metrics catch clear-cut errors, but some quality aspects require judgment:

- Was tool selection *appropriate* even if technically valid? (e.g., looking up an order before initiating a return is good practice even if not strictly required)
- Was the tool sequence *efficient*? (e.g., calling the same tool twice with different params vs. once with the right params)
- Was the final response *helpful* given the tool results?

An LLM judge can assess these subjective dimensions using a structured rubric.

In [ ]:
def judge_trajectory(test_case, result):
    """Use an LLM judge to evaluate the quality of a tool-calling trajectory.

    Evaluates 5 dimensions on a 1-5 scale, returns structured scores.
    """
    # Build tool descriptions for the judge
    tool_descriptions = "\n".join([
        f"- {t['name']}: {t['description']}"
        for t in tool_defs['tools']
    ])

    trajectory_str = json.dumps(result['trajectory'], indent=2) if result['trajectory'] else "(no tools called)"

    judge_prompt = f"""You are an expert evaluator of AI agent tool-calling behavior. Evaluate the following agent trajectory for a customer support scenario.

<available_tools>
{tool_descriptions}
</available_tools>

<user_request>
{test_case['input']}
</user_request>

<expected_tools>
{json.dumps(test_case['expected_tools'])}
</expected_tools>

<actual_trajectory>
{trajectory_str}
</actual_trajectory>

<final_response>
{result['final_output'][:500]}
</final_response>

Evaluate on these 5 dimensions (score each 1-5):

1. **Tool Selection**: Did the agent pick the right tools for the task? (5=perfect, 1=completely wrong tools or unnecessary calls)
2. **Parameter Quality**: Were parameters correct, complete, and well-formed? (5=all correct, 1=hallucinated or missing)
3. **Sequence Logic**: Were tools called in a sensible, efficient order? (5=optimal order, 1=illogical sequence)
4. **Efficiency**: Were unnecessary calls avoided? Did the agent use the minimum tools needed? (5=minimal and sufficient, 1=wasteful)
5. **Response Quality**: Did the final response appropriately address the user's request given the tool results? (5=excellent, 1=unhelpful)

Return ONLY valid JSON in this exact format:
{{
    "tool_selection": {{"score": <1-5>, "reason": "<brief explanation>"}},
    "parameter_quality": {{"score": <1-5>, "reason": "<brief explanation>"}},
    "sequence_logic": {{"score": <1-5>, "reason": "<brief explanation>"}},
    "efficiency": {{"score": <1-5>, "reason": "<brief explanation>"}},
    "response_quality": {{"score": <1-5>, "reason": "<brief explanation>"}},
    "overall_assessment": "<one sentence summary>"
}}"""

    response = bedrock.converse(
        modelId=JUDGE_MODEL,
        messages=[{"role": "user", "content": [{"text": judge_prompt}]}],
        system=[{"text": "You are an evaluation expert. Return only valid JSON, no other text."}]
    )

    # Extract and parse the judge's response
    judge_text = response['output']['message']['content'][0]['text']

    # Try to extract JSON from the response
    try:
        # Handle potential markdown code blocks
        if '```' in judge_text:
            judge_text = judge_text.split('```')[1]
            if judge_text.startswith('json'):
                judge_text = judge_text[4:]
        scores = json.loads(judge_text.strip())
    except json.JSONDecodeError:
        # Fallback: try to find JSON object in the text
        match = re.search(r'\{[\s\S]*\}', judge_text)
        if match:
            scores = json.loads(match.group())
        else:
            scores = {"error": "Failed to parse judge response", "raw": judge_text[:200]}

    return scores

### Run LLM-as-Judge on Mock Evaluation Results

We'll judge a subset of test cases — the ones that are most interesting (hard cases, edge cases, and known failures).

In [ ]:
# Select interesting test cases for judging (mix of difficulties)
judge_test_ids = ["tc-001", "tc-002", "tc-004", "tc-005", "tc-008", "tc-009", "tc-012"]

print("LLM-as-Judge Evaluation")
print("=" * 90)

judge_results = []
for tc_id in judge_test_ids:
    tc = tc_lookup[tc_id]

    # Re-run with mock tools (or use cached results if available)
    result = run_with_mock_tools(tc['input'])

    # Get judge evaluation
    scores = judge_trajectory(tc, result)

    if 'error' not in scores:
        dims = ['tool_selection', 'parameter_quality', 'sequence_logic', 'efficiency', 'response_quality']
        avg_score = sum(scores[d]['score'] for d in dims) / len(dims)

        print(f"\n  {tc_id} - {tc['name']}")
        print(f"    Tools: {result['tools_called']}")
        for dim in dims:
            print(f"    {dim:20s}: {scores[dim]['score']}/5 - {scores[dim]['reason']}")
        print(f"    Average: {avg_score:.1f}/5")
        print(f"    Assessment: {scores.get('overall_assessment', 'N/A')}")

        judge_results.append({
            'test_id': tc_id,
            'name': tc['name'][:30],
            **{d: scores[d]['score'] for d in dims},
            'average': round(avg_score, 2),
            'tools': str(result['tools_called'])
        })
    else:
        print(f"\n  {tc_id} - JUDGE ERROR: {scores.get('error', 'unknown')}")

In [ ]:
# Summary of LLM-as-Judge results
if judge_results:
    df_judge = pd.DataFrame(judge_results)
    dims = ['tool_selection', 'parameter_quality', 'sequence_logic', 'efficiency', 'response_quality']

    print("\nLLM-as-Judge Summary")
    print("=" * 70)
    for dim in dims:
        print(f"  {dim:20s}: {df_judge[dim].mean():.1f}/5 avg")
    print(f"  {'OVERALL':20s}: {df_judge['average'].mean():.1f}/5 avg")

    print(f"\nDetailed Scores:")
    print(df_judge[['test_id', 'name'] + dims + ['average']].to_string(index=False))

## 7. Approach 5: Multi-Turn Simulation (Cost: $$$)

Real agent conversations are rarely single-turn. A customer might:
1. Ask about an order
2. Follow up with a return request based on the order details
3. Ask about the return policy
4. Request escalation if unsatisfied

Multi-turn simulation tests whether the agent maintains context and makes correct tool decisions across a full conversation session.

### How It Works

We define **conversation scenarios** — sequences of user messages that simulate realistic interactions. The agent runs with mock tools for each turn, and we evaluate the complete session trajectory.

In [ ]:
# Define multi-turn conversation scenarios
MULTI_TURN_SCENARIOS = [
    {
        "id": "mt-001",
        "name": "Order lookup then return",
        "description": "Customer checks an order, then decides to return it",
        "turns": [
            {
                "user": "Hi, can you check the status of my order ORD-48291?",
                "expected_tools": ["lookup_order"],
                "description": "Initial order inquiry"
            },
            {
                "user": "Actually, I want to return the headphones from that order. They're defective.",
                "expected_tools": ["initiate_return"],
                "description": "Follow-up return request (should use context from previous lookup)"
            }
        ],
        "goal": "Customer should get order details and then successfully initiate a return"
    },
    {
        "id": "mt-002",
        "name": "General question then specific action",
        "description": "Customer asks a general question, then makes a specific request",
        "turns": [
            {
                "user": "What's your return policy?",
                "expected_tools": [],
                "description": "General question - no tool needed"
            },
            {
                "user": "OK, then I'd like to return order ORD-33012. I changed my mind about it.",
                "expected_tools": ["initiate_return"],
                "description": "Specific return request"
            }
        ],
        "goal": "Agent answers policy question without tools, then handles return with tools"
    },
    {
        "id": "mt-003",
        "name": "Complex multi-step with escalation",
        "description": "Customer has a complex issue requiring multiple tools and eventual escalation",
        "turns": [
            {
                "user": "I'm customer CUST-4421. Can you pull up my profile?",
                "expected_tools": ["get_customer_profile"],
                "description": "Profile lookup"
            },
            {
                "user": "I need to check on order ORD-67890 and also see if the BT-Speaker-Pro is back in stock.",
                "expected_tools": ["lookup_order", "check_inventory"],
                "description": "Multiple tool calls in one turn"
            },
            {
                "user": "The order was wrong and I've been waiting weeks for a refund. I want to speak to a manager.",
                "expected_tools": ["escalate_to_human"],
                "description": "Escalation request"
            }
        ],
        "goal": "Agent handles profile lookup, parallel inquiries, and escalation across the session"
    }
]

print(f"Defined {len(MULTI_TURN_SCENARIOS)} multi-turn scenarios")
for s in MULTI_TURN_SCENARIOS:
    print(f"  {s['id']}: {s['name']} ({len(s['turns'])} turns)")

In [ ]:
def run_multi_turn_scenario(scenario):
    """Run a multi-turn conversation with mock tools and evaluate each turn.

    Maintains conversation context across turns, just like a real agent would.
    """
    messages = []
    all_trajectories = []
    turn_results = []

    for i, turn in enumerate(scenario['turns']):
        # Add user message
        messages.append({"role": "user", "content": [{"text": turn['user']}]})

        turn_trajectory = []

        # Run agent (may take multiple API calls if tools are involved)
        for _ in range(5):  # max sub-turns per user message
            response = bedrock.converse(
                modelId=AGENT_MODEL,
                messages=messages,
                system=[{"text": SYSTEM_PROMPT}],
                toolConfig=tool_config
            )

            stop_reason = response['stopReason']
            assistant_content = response['output']['message']['content']
            messages.append({"role": "assistant", "content": assistant_content})

            if stop_reason == 'end_turn':
                final_text = ""
                for block in assistant_content:
                    if 'text' in block:
                        final_text += block['text']
                break

            if stop_reason == 'tool_use':
                tool_results = []
                for block in assistant_content:
                    if 'toolUse' in block:
                        tool_call = block['toolUse']
                        turn_trajectory.append({
                            'tool': tool_call['name'],
                            'params': tool_call['input']
                        })

                        mock_fn = MOCK_RESPONSES.get(tool_call['name'])
                        mock_result = mock_fn(tool_call['input']) if mock_fn else {"error": "Unknown tool"}

                        tool_results.append({
                            "toolResult": {
                                "toolUseId": tool_call['toolUseId'],
                                "content": [{"json": mock_result}],
                                "status": "success"
                            }
                        })

                messages.append({"role": "user", "content": tool_results})

        # Evaluate this turn
        actual_tools = [t['tool'] for t in turn_trajectory]
        expected_tools = set(turn['expected_tools'])
        actual_set = set(actual_tools)

        tp = len(expected_tools & actual_set)
        fp = len(actual_set - expected_tools)
        fn = len(expected_tools - actual_set)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 1.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        turn_results.append({
            'turn': i + 1,
            'description': turn['description'],
            'expected_tools': turn['expected_tools'],
            'actual_tools': actual_tools,
            'f1': round(f1, 3),
            'response_preview': final_text[:100] if 'final_text' in dir() else ""
        })

        all_trajectories.extend(turn_trajectory)

    return {
        'scenario_id': scenario['id'],
        'name': scenario['name'],
        'turn_results': turn_results,
        'full_trajectory': all_trajectories,
        'total_tools_called': len(all_trajectories),
        'messages': messages
    }

In [ ]:
# Run all multi-turn scenarios
print("Multi-Turn Simulation Results")
print("=" * 90)

mt_results = []
for scenario in MULTI_TURN_SCENARIOS:
    print(f"\nScenario: {scenario['name']}")
    print(f"  Goal: {scenario['goal']}")

    result = run_multi_turn_scenario(scenario)
    mt_results.append(result)

    session_f1_scores = []
    for tr in result['turn_results']:
        status = "PASS" if tr['f1'] == 1.0 else "PARTIAL" if tr['f1'] > 0 else "FAIL"
        expected_str = ", ".join(tr['expected_tools']) if tr['expected_tools'] else "(none)"
        actual_str = ", ".join(tr['actual_tools']) if tr['actual_tools'] else "(none)"
        print(f"  Turn {tr['turn']} [{status}]: {tr['description']}")
        print(f"    Expected: [{expected_str}]  Actual: [{actual_str}]  F1={tr['f1']}")
        session_f1_scores.append(tr['f1'])

    avg_f1 = sum(session_f1_scores) / len(session_f1_scores) if session_f1_scores else 0
    print(f"  Session Average F1: {avg_f1:.3f}")
    print(f"  Total tool calls across session: {result['total_tools_called']}")

### Goal-Based Evaluation with LLM Judge

Beyond per-turn metrics, we can ask an LLM judge whether the agent achieved the overall goal of each scenario.

In [ ]:
def judge_goal_success(scenario, result):
    """Use an LLM judge to evaluate whether the agent achieved the scenario's goal."""

    # Build conversation summary
    conversation_summary = []
    for tr in result['turn_results']:
        tools_str = ", ".join(tr['actual_tools']) if tr['actual_tools'] else "none"
        conversation_summary.append(
            f"Turn {tr['turn']}: User asked about '{tr['description']}' -> Agent used tools: [{tools_str}]"
        )

    judge_prompt = f"""Evaluate whether an AI customer support agent achieved the user's goal in a multi-turn conversation.

<scenario_goal>
{scenario['goal']}
</scenario_goal>

<conversation_summary>
{chr(10).join(conversation_summary)}
</conversation_summary>

<full_tool_trajectory>
{json.dumps([{'tool': t['tool'], 'params': t['params']} for t in result['full_trajectory']], indent=2)}
</full_tool_trajectory>

Evaluate:
1. **Goal Achievement** (1-5): Did the agent accomplish what the user needed?
2. **Context Maintenance** (1-5): Did the agent properly use information from earlier turns?
3. **Tool Efficiency** (1-5): Were the right tools used at the right times across the session?

Return ONLY valid JSON:
{{
    "goal_achievement": {{"score": <1-5>, "reason": "<explanation>"}},
    "context_maintenance": {{"score": <1-5>, "reason": "<explanation>"}},
    "tool_efficiency": {{"score": <1-5>, "reason": "<explanation>"}},
    "goal_met": <true/false>,
    "summary": "<one sentence>"
}}"""

    response = bedrock.converse(
        modelId=JUDGE_MODEL,
        messages=[{"role": "user", "content": [{"text": judge_prompt}]}],
        system=[{"text": "You are an evaluation expert. Return only valid JSON."}]
    )

    judge_text = response['output']['message']['content'][0]['text']
    try:
        if '```' in judge_text:
            judge_text = judge_text.split('```')[1]
            if judge_text.startswith('json'):
                judge_text = judge_text[4:]
        return json.loads(judge_text.strip())
    except json.JSONDecodeError:
        match = re.search(r'\{[\s\S]*\}', judge_text)
        return json.loads(match.group()) if match else {"error": "Parse failed"}

# Run goal evaluation on all scenarios
print("\nGoal-Based Evaluation")
print("=" * 90)

for i, scenario in enumerate(MULTI_TURN_SCENARIOS):
    result = mt_results[i]
    goal_eval = judge_goal_success(scenario, result)

    print(f"\n  {scenario['id']}: {scenario['name']}")
    if 'error' not in goal_eval:
        print(f"    Goal Met: {'Yes' if goal_eval.get('goal_met') else 'No'}")
        for dim in ['goal_achievement', 'context_maintenance', 'tool_efficiency']:
            if dim in goal_eval:
                print(f"    {dim:25s}: {goal_eval[dim]['score']}/5 - {goal_eval[dim]['reason']}")
        print(f"    Summary: {goal_eval.get('summary', 'N/A')}")
    else:
        print(f"    Error: {goal_eval['error']}")

## 8. Approach 6: Strands ToolSimulator + ActorSimulator (Cost: $$$$)

Approaches 3-5 above used **hardcoded mock responses** — deterministic lambdas that always return the same data. This works well for testing specific scenarios, but has limitations:

- Mock responses don't adapt to the conversation context
- You have to manually write a mock for every tool
- The same `lookup_order` call always returns the same items, regardless of the order ID
- Cross-tool consistency is your responsibility (does `initiate_return` "know" what `lookup_order` returned?)

The **Strands Evals SDK** solves this with two simulators:

### ToolSimulator: LLM-Powered Tool Responses

Instead of hardcoded mocks, `ToolSimulator` uses an LLM to generate realistic tool responses:

1. You register tools with `@simulator.tool(output_schema=MySchema)` — same decorator pattern as `@tool`
2. When the agent calls a tool, the simulator intercepts it and **prompts an LLM** to generate a schema-compliant response
3. A `StateRegistry` caches all previous calls, so responses are **consistent across the conversation** (if order ORD-48291 had headphones in it, the return tool "knows" that)
4. Tools can `share_state_id` so related tools share the same state context

### ActorSimulator: LLM-Powered User Simulation

Instead of scripted user turns, `ActorSimulator` generates dynamic, goal-driven user behavior:

1. Given a `Case` with a `task_description`, it auto-generates a realistic user persona
2. The simulated user adapts to the agent's responses (asks follow-ups, provides clarification, etc.)
3. Built-in goal tracking — the simulator knows when the objective is met and stops

### Combined: Fully Synthetic Evaluation

```
┌─────────────────────────────────────────────────────────┐
│                  No Real Anything                        │
│                                                         │
│   ActorSimulator ──> Agent ──> ToolSimulator            │
│   (fake user)        (real)    (fake tools, LLM-powered)│
│                                                         │
│   User messages:     Decisions:    Tool responses:      │
│   LLM-generated      Real model    LLM-generated,       │
│   goal-driven        under test    schema-compliant,     │
│                                    stateful              │
└─────────────────────────────────────────────────────────┘
```

### Step 1: Define Tools with ToolSimulator

We register the same customer support tools using `@simulator.tool()`. Each tool gets a Pydantic `output_schema` that tells the LLM what shape the response should take. The function body is `pass` — it never executes. The LLM generates responses based on the schema, docstring, and accumulated state.

Tools that share state (e.g., `lookup_order` and `initiate_return` both deal with orders) use `share_state_id` so the simulator maintains consistency across them.

In [ ]:
from strands_evals.simulation import ToolSimulator
from strands.models import BedrockModel
from pydantic import BaseModel, Field
from typing import Optional
from botocore.config import Config

# ToolSimulator uses an LLM to generate tool responses.
# It needs its own model — can be a cheaper/faster model than the agent under test.
SIMULATOR_MODEL = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    boto_client_config=Config(connect_timeout=5, read_timeout=60, retries={"max_attempts": 2})
)

simulator = ToolSimulator(model=SIMULATOR_MODEL)

# --- Output Schemas (Pydantic models defining what each tool returns) ---

class OrderResponse(BaseModel):
    order_id: str = Field(description="The order ID")
    status: str = Field(description="Order status: pending, shipped, delivered, cancelled")
    items: list[str] = Field(description="Items in the order")
    delivered_date: Optional[str] = Field(default=None, description="Delivery date if delivered")
    total: str = Field(description="Order total")
    payment_status: str = Field(default="charged", description="Payment status")

class ReturnResponse(BaseModel):
    return_auth: str = Field(description="Return authorization number")
    instructions: str = Field(description="Return shipping instructions")
    refund_estimate: str = Field(default="5-7 business days", description="Estimated refund timeline")

class InventoryResponse(BaseModel):
    product_id: str = Field(description="Product ID checked")
    available: bool = Field(description="Whether the product is in stock")
    quantity: int = Field(description="Available quantity")
    warehouse: str = Field(description="Warehouse region")
    estimated_delivery: str = Field(default="2-3 business days", description="Estimated delivery time")

class AddressUpdateResponse(BaseModel):
    status: str = Field(description="Update status: updated or error")
    order_id: str = Field(description="The order ID")
    note: str = Field(description="Additional information about the update")

class CustomerProfileResponse(BaseModel):
    customer_id: str = Field(description="Customer ID")
    name: str = Field(description="Customer name")
    tier: str = Field(description="Loyalty tier: Bronze, Silver, Gold, Platinum")
    total_orders: int = Field(description="Total number of orders")

class EscalationResponse(BaseModel):
    ticket: str = Field(description="Escalation ticket number")
    estimated_wait: str = Field(description="Estimated wait time")
    priority: str = Field(description="Priority level assigned")

# --- Register tools with the simulator ---
# share_state_id groups tools that should see each other's history
# initial_state_description seeds the LLM with context about the simulated environment

ORDER_STATE = "order_management"
CUSTOMER_STATE = "customer_data"

@simulator.tool(
    output_schema=OrderResponse,
    share_state_id=ORDER_STATE,
    initial_state_description="E-commerce order database. Orders use format ORD-XXXXX. Statuses: pending, shipped, delivered. Most orders contain 1-3 items."
)
def lookup_order(order_id: str) -> dict:
    """Look up order details by order ID. Returns order status, items, shipping info, and payment summary."""
    pass

@simulator.tool(
    output_schema=ReturnResponse,
    share_state_id=ORDER_STATE,
    initial_state_description="Return processing system. Returns require order_id and reason. Auth numbers use format RET-XXXXX."
)
def initiate_return(order_id: str, reason: str, items: list[str] = None) -> dict:
    """Start a return process for a specific order. Requires order ID and reason."""
    pass

@simulator.tool(output_schema=InventoryResponse)
def check_inventory(product_id: str, warehouse_region: str = "us-east") -> dict:
    """Check if a product is currently in stock."""
    pass

@simulator.tool(output_schema=AddressUpdateResponse, share_state_id=ORDER_STATE)
def update_shipping_address(order_id: str, new_address: dict) -> dict:
    """Update the shipping address for an order that has not yet shipped."""
    pass

@simulator.tool(
    output_schema=CustomerProfileResponse,
    share_state_id=CUSTOMER_STATE,
    initial_state_description="Customer database. Customer IDs use format CUST-XXXX. Tiers: Bronze, Silver, Gold, Platinum."
)
def get_customer_profile(customer_id: str) -> dict:
    """Retrieve customer profile information including order history summary and loyalty tier."""
    pass

@simulator.tool(output_schema=EscalationResponse)
def escalate_to_human(reason: str, priority: str = "medium", context_summary: str = None) -> dict:
    """Transfer the conversation to a human support agent."""
    pass

print(f"Registered {len(simulator.list_tools())} tools with ToolSimulator: {simulator.list_tools()}")

### Step 2: Create Agent with Simulated Tools

The agent uses `simulator.get_tool()` to get wrapped versions of each tool. These wrappers have the same name, description, and input schema as the real tools — the agent can't tell the difference. But when it calls them, the ToolSimulator's LLM generates the response instead of real backend execution.

In [ ]:
from strands import Agent

# Get simulated tool wrappers — same interface as real tools, but LLM-powered responses
simulated_tools = [simulator.get_tool(name) for name in simulator.list_tools()]

# Create the agent under test with simulated tools
AGENT_BEDROCK_MODEL = BedrockModel(
    model_id=AGENT_MODEL,
    boto_client_config=Config(connect_timeout=5, read_timeout=60, retries={"max_attempts": 2})
)

agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    tools=simulated_tools,
    model=AGENT_BEDROCK_MODEL,
    callback_handler=None
)

print(f"Agent created with {len(simulated_tools)} simulated tools")
print("The agent will make real tool-calling decisions, but all tool execution is simulated via LLM")

### Step 3: Single-Turn Test with ToolSimulator

Before combining with ActorSimulator, let's verify the ToolSimulator works. Notice how the LLM generates realistic, schema-compliant responses — and how the StateRegistry tracks calls for consistency.

In [ ]:
# Single-turn test: lookup an order, then return it
# The ToolSimulator should generate consistent details across both calls

test_input = "I received the wrong item in order ORD-55123. Can you look it up and start a return?"
print(f"Input: {test_input}\n")

response = agent(test_input)
print(f"Agent Response:\n{response}\n")

# Inspect the StateRegistry to see what the simulator generated
print("=" * 70)
print("ToolSimulator State (order_management):")
order_state = simulator.get_state(ORDER_STATE)
for call in order_state.get('previous_calls', []):
    print(f"\n  Tool: {call['tool_name']}")
    print(f"  Params: {json.dumps(call['parameters'], indent=4)}")
    print(f"  Response: {json.dumps(call['response'], indent=4)}")

# Check the agent's tool metrics
print(f"\nAgent Tool Metrics:")
if hasattr(response, 'metrics') and hasattr(response.metrics, 'tool_metrics'):
    for tool_name, metrics in response.metrics.tool_metrics.items():
        if metrics.call_count > 0:
            print(f"  {tool_name}: {metrics.call_count} call(s)")

### Step 4: Fully Synthetic Evaluation — ToolSimulator + ActorSimulator

Now we combine both simulators. The `ActorSimulator` generates realistic user messages, the agent processes them with `ToolSimulator`-powered tools, and we evaluate the full session.

This is the most comprehensive approach: **no real tools, no real users, no scripted scenarios** — yet it tests realistic multi-turn interactions with stateful tool responses.

In [ ]:
from strands_evals import ActorSimulator, Case

# Define evaluation scenarios as Cases
# The task_description drives the ActorSimulator's goal-tracking
synthetic_cases = [
    Case(
        name="order-return-flow",
        input="Hi, can you check the status of my order ORD-48291?",
        metadata={
            "task_description": "Customer checks order status, discovers an issue with the items, and successfully initiates a return",
            "category": "multi_step"
        }
    ),
    Case(
        name="inventory-and-purchase",
        input="Is the BT-Speaker-Pro available? I'm in the US West region.",
        metadata={
            "task_description": "Customer checks product availability and gets stock information for their region",
            "category": "single_step"
        }
    ),
    Case(
        name="complex-escalation",
        input="I'm customer CUST-8832 and I have a serious billing issue with my recent order.",
        metadata={
            "task_description": "Customer profile is looked up, billing issue is investigated, and the case is escalated to a human agent with full context",
            "category": "escalation"
        }
    ),
]

print(f"Defined {len(synthetic_cases)} synthetic evaluation cases:")
for case in synthetic_cases:
    print(f"  {case.name}: {case.metadata['task_description'][:70]}...")

In [ ]:
def run_synthetic_evaluation(case, max_turns=6):
    """Run a fully synthetic evaluation: ActorSimulator (user) + ToolSimulator (tools).

    Neither real tools nor real users are involved. The agent's tool-calling
    decisions are the only real thing being tested.
    """
    # Clear simulator state for a fresh scenario
    simulator.state_registry.clear_state(ORDER_STATE)
    simulator.state_registry.clear_state(CUSTOMER_STATE)

    # Create ActorSimulator — it auto-generates a user persona from the Case
    user_sim = ActorSimulator.from_case_for_user_simulator(
        case=case,
        max_turns=max_turns
    )

    # Create a fresh agent for this scenario
    agent = Agent(
        system_prompt=SYSTEM_PROMPT,
        tools=[simulator.get_tool(name) for name in simulator.list_tools()],
        model=AGENT_BEDROCK_MODEL,
        callback_handler=None
    )

    # Run the conversation loop
    user_message = case.input
    conversation = []
    tools_used_per_turn = []

    turn = 0
    while user_sim.has_next():
        turn += 1

        # Agent responds to user message
        agent_response = agent(user_message)
        agent_message = str(agent_response)

        # Capture tools used in this turn
        turn_tools = []
        if hasattr(agent_response, 'metrics') and hasattr(agent_response.metrics, 'tool_metrics'):
            for tool_name, metrics in agent_response.metrics.tool_metrics.items():
                if metrics.call_count > 0:
                    turn_tools.extend([tool_name] * metrics.call_count)

        tools_used_per_turn.append(turn_tools)
        conversation.append({
            "turn": turn,
            "user": user_message,
            "agent": agent_message[:200],
            "tools_used": turn_tools
        })

        # Simulated user responds (with reasoning about the conversation)
        user_result = user_sim.act(agent_message)
        user_message = str(user_result.structured_output.message)

        # Log the simulated user's reasoning
        conversation[-1]["user_reasoning"] = str(user_result.structured_output.reasoning)[:150]

    return {
        "case_name": case.name,
        "total_turns": turn,
        "conversation": conversation,
        "all_tools": [t for turn_tools in tools_used_per_turn for t in turn_tools],
        "tools_per_turn": tools_used_per_turn,
        "final_state": {
            "order_state": simulator.get_state(ORDER_STATE),
            "customer_state": simulator.get_state(CUSTOMER_STATE),
        }
    }

print("Synthetic evaluation function ready")

In [ ]:
# Run fully synthetic evaluation on all cases
print("Fully Synthetic Evaluation (ToolSimulator + ActorSimulator)")
print("=" * 90)
print("No real tools. No real users. No scripted scenarios.\n")

synthetic_results = []
for case in synthetic_cases:
    print(f"\nCase: {case.name}")
    print(f"  Goal: {case.metadata['task_description']}")

    try:
        result = run_synthetic_evaluation(case)
        synthetic_results.append(result)

        print(f"  Turns: {result['total_turns']}")
        print(f"  All tools used: {result['all_tools']}")
        print(f"\n  Conversation Flow:")
        for entry in result['conversation']:
            tools_str = ", ".join(entry['tools_used']) if entry['tools_used'] else "(none)"
            print(f"    Turn {entry['turn']}:")
            print(f"      User: {entry['user'][:80]}...")
            print(f"      Tools: [{tools_str}]")
            print(f"      Agent: {entry['agent'][:80]}...")
            if entry.get('user_reasoning'):
                print(f"      Sim reasoning: {entry['user_reasoning'][:80]}...")

    except Exception as e:
        print(f"  ERROR: {str(e)[:100]}")

print(f"\n\nCompleted {len(synthetic_results)}/{len(synthetic_cases)} synthetic evaluations")

### Hardcoded Mocks vs. ToolSimulator — When to Use Each

| Dimension | Hardcoded Mocks (Approach 3) | ToolSimulator (Approach 6) |
|-----------|------------------------------|---------------------------|
| **Setup effort** | Write a mock lambda per tool | Define a Pydantic schema per tool |
| **Response quality** | You control exactly what comes back | LLM generates realistic, varied responses |
| **Consistency** | Manual (your mocks must be coherent) | Automatic via StateRegistry |
| **Determinism** | Fully deterministic | Non-deterministic (LLM-generated) |
| **Cost per tool call** | $0 (no API call) | $$ (LLM call per tool invocation) |
| **Best for** | CI gates, regression tests, exact scenarios | Exploratory testing, discovering edge cases |

**Recommendation**: Use hardcoded mocks for **repeatable CI/CD testing** where you need deterministic results. Use ToolSimulator for **exploratory evaluation** where you want to discover how the agent handles varied, realistic tool responses it hasn't seen before.

## 9. Putting It All Together: Choosing Your Evaluation Approach

### Decision Framework

Each approach has a sweet spot. Use them in combination for comprehensive evaluation:

| When to Use | Approach | Why |
|-------------|----------|-----|
| **Every commit / CI pipeline** | Static Analysis + Schema Validation | Free, instant, catches regressions |
| **Pre-deployment testing** | Mock Tool Evaluation | Tests live model behavior safely |
| **Quality deep-dives** | LLM-as-Judge | Catches subjective quality issues |
| **Release validation** | Multi-Turn Simulation | Tests end-to-end goal achievement |
| **Exploratory / edge-case discovery** | Strands ToolSimulator + ActorSimulator | Fully synthetic, stateful, no scripting needed |

### Recommended Evaluation Pipeline

```
Production Logs ─────> [1] Static Analysis ─────> Dashboard/Alerts
                              │
New Model Version ───> [2] Schema Validation ──> Gate: block if < 95% compliance
                              │
                       [3] Mock Tool Eval ─────> Gate: block if F1 < 0.9
                              │
                       [4] LLM-as-Judge ───────> Report: flag scores < 3/5
                              │
Pre-Release ──────────> [5] Multi-Turn Sim ────> Gate: block if goal success < 80%
                              │
Edge-Case Discovery ──> [6] Synthetic Sim ─────> Discover: new failure modes
```

### The Two Halves of "No Real Tool Calls"

This module demonstrates two fundamentally different ways to avoid real tool execution:

| Strategy | Approaches | How It Works |
|----------|-----------|--------------|
| **Hardcoded Mocks** | 3, 4, 5 | You write deterministic mock responses. Cheap, fast, repeatable. |
| **LLM-Powered Simulation** | 6 | An LLM generates realistic, stateful tool responses. Varied, realistic, discovers edge cases. |

Both are valid. Use hardcoded mocks for **CI gates and regression tests** where determinism matters. Use LLM-powered simulation for **exploratory testing** where you want to discover how the agent handles varied responses it hasn't seen before.

### Cost Comparison

For a test suite of 100 cases:

| Approach | API Calls | Estimated Cost | Time |
|----------|-----------|---------------|------|
| Static Analysis | 0 | $0.00 | < 1 second |
| Schema Validation | 0 | $0.00 | < 1 second |
| Mock Tool Eval | ~100-300 | ~$0.50-2.00 | 2-5 minutes |
| LLM-as-Judge | ~100 | ~$0.50-1.00 | 2-3 minutes |
| Multi-Turn Sim | ~300-1000 | ~$2.00-8.00 | 10-20 minutes |
| Strands Synthetic Sim | ~500-2000 | ~$5.00-20.00 | 15-30 minutes |

### What Each Approach Misses

No single approach catches everything:

- **Static Analysis** misses: errors the model makes on new inputs (only evaluates recorded traces)
- **Schema Validation** misses: semantically wrong but structurally valid params (e.g., wrong `order_id` that matches the pattern)
- **Mock Tool Eval** misses: subjective quality issues, multi-turn context problems
- **LLM-as-Judge** misses: can be inconsistent, may miss structural errors that programmatic checks catch
- **Multi-Turn Sim** misses: rare edge cases not covered by scenarios (expensive to scale)
- **Strands Synthetic Sim** misses: non-deterministic (hard to reproduce exact failures), expensive to run at scale

**Use them together.** The pyramid works because each layer catches what the layer below misses.

## 10. Key Takeaways

### Core Principle
**Separate what you're evaluating from what you're executing.** You're evaluating the model's *decisions* (whether, which, what). You don't need real tools for that.

### Six Approaches, One Pipeline
1. **Static Trajectory Analysis** ($0) — your first line of defense, run on every trace
2. **Schema Validation** ($0) — catches parameter hallucination automatically
3. **Mock Tool Evaluation** ($) — tests live model behavior without side effects
4. **LLM-as-Judge** ($$) — adds subjective quality assessment
5. **Multi-Turn Simulation** ($$$) — validates end-to-end goal achievement with scripted scenarios
6. **Strands ToolSimulator + ActorSimulator** ($$$$) — fully synthetic, stateful, no scripting needed

### Key Metrics to Track
- **Tool Selection F1** — the single most important metric for tool-calling agents
- **Call/No-Call Accuracy** — agents that over-call tools waste resources and confuse users
- **Schema Compliance** — your cheapest defense against parameter hallucination
- **Sequence Match** — especially important for multi-step workflows
- **Goal Success Rate** — the ultimate measure for multi-turn agents
- **State Consistency** — are tool responses coherent across calls? (ToolSimulator tracks this automatically)

### Production Recommendations
1. Run static analysis + schema validation on **every production trace** (it's free)
2. Run mock tool evaluation as a **CI gate** on model or prompt changes
3. Use LLM-as-Judge for **periodic quality audits** and edge case investigation
4. Run multi-turn simulation for **release validation** of critical workflows
5. Use Strands synthetic simulation for **exploratory testing** and discovering unknown failure modes
6. Track metrics over time — tool-calling quality can degrade with model updates

### Next Steps
- Adapt the mock tool layer to your own agent's tools
- Build a ground truth test suite from your production logs
- Set up automated evaluation in your CI/CD pipeline
- Try ToolSimulator with your own tools — define Pydantic schemas and let the LLM generate responses
- Use ActorSimulator to discover edge cases your scripted scenarios don't cover